In [10]:
import onnxruntime
import lightgbm as lgb
# import matplotlib.pyplot as plt
from sklearn.datasets import load_svmlight_file
from sklearn.ensemble import GradientBoostingClassifier
import onnxmltools
import fastinference.Loader
import fastinference
import pprint
#

In [2]:

# load lightgbm tree booster from txt file
booster = lgb.Booster(model_file='C:/Users/Jan Stenkamp/Documents/Arbeit/Boosted Trees/code/win/LightGBM/experiments/models/covtype/data-covtypems-64000-fp-0.03125-tp-0.03125-tree-5-depth-3.txt')
# booster = lgb.Booster(model_file='C:/Users/Jan Stenkamp/Documents/Arbeit/Boosted Trees/code/win/LightGBM/experiments/models/california_housing/data-california_housingms-64000-fp-1.0-tp-32.0-tree-5-depth-3.txt')
# dump = booster.dump_model()

# import json
# with open('model.json', 'w') as fp:
#     json.dump(dump, fp)

# convert to onnx
# TODO: what does this type do? FloatTensorType([None, 54]): first dimension is batch size, second dimension is number of features?
# onnx_model = onnxmltools.convert_lightgbm(booster, initial_types=[('float_input', onnxmltools.convert.common.data_types.FloatTensorType([None, 54]))])
# onnxmltools.utils.save_model(onnx_model, 'model.onnx')



[LightGBM] [Warning] Ignoring unrecognized parameter 'tinygbdt_penalty_feature' found in model string.
[LightGBM] [Warning] Ignoring unrecognized parameter 'tinygbdt_penalty_split' found in model string.
[LightGBM] [Warning] Ignoring unrecognized parameter 'tinygbdt_forestsize' found in model string.


In [3]:
X_train, y_train = load_svmlight_file('C:/Users/Jan Stenkamp/Documents/Arbeit/Boosted Trees/code/win/LightGBM/experiments/data/covtype.train')
X_test, y_test = load_svmlight_file('C:/Users/Jan Stenkamp/Documents/Arbeit/Boosted Trees/code/win/LightGBM/experiments/data/covtype.test')

# skmodel = lgb.LGBMClassifier(predict_disable_shape_check=True)
# skmodel.fit(X_train, y_train, init_model=booster)#, predict_disable_shape_check=True)
skmodel = GradientBoostingClassifier(n_estimators=2, init='zero')
# TODO: try HistGradientBoostingClassifier -> probably not supported by fastinference model loader 
skmodel.fit(X_test, y_test)

# load libsvm data for training as numpy array
print(skmodel.predict(X_train))
print(y_train)


[0. 1. 1. ... 0. 1. 1.]
[0. 0. 1. ... 0. 1. 0.]


In [4]:
isinstance(skmodel, GradientBoostingClassifier)

True

In [5]:
# access single column in sparse matrix format
X_test.shape

(116203, 54)

In [6]:

# ensemble = fastinference.Loader.model_from_sklearn(skmodel)
# ensemble = fastinference.Loader.model_from_sklearn(skmodel, name="model", accuracy=None)
ensemble = fastinference.Loader.model_from_sklearn(skmodel, name="model", accuracy=None)
# ensemble = fastinference.models.Ensemble.Ensemble.from_sklearn(skmodel)
# 
# test = GradientBoostingClassifier()
# ensemble = fastinference.Loader.model_from_sklearn(test, name="model", accuracy=None)


In [7]:
ensemble.optimize("pruning", {"pruning_method": "reduced_error", "x_prune": X_test, "y_prune": y_test}, None, None)
# loaded_model.implement("/my/nice/model", "model", "my.newest.implementation")

[0.0, 1.0]


c:\users\jan stenkamp\documents\arbeit\boosted trees\code\win\lightgbm\experiments\baselines\fastinference\fastinference\models\Tree.py:99: SparseEfficiencyWarning: Comparing a sparse matrix with a scalar greater than zero using <= is inefficient, try using > instead.
  if (x[:,node.feature] <= node.split):


In [11]:
pprint.pp(ensemble.to_dict())

{'classes': [0.0, 1.0],
 'n_classes': 2,
 'n_features': 54,
 'category': 'ensemble',
 'accuracy': None,
 'name': 'model',
 'models': [{'weight': 0.5,
             'model': {'classes': [0.0, 1.0],
                       'n_classes': 2,
                       'n_features': 54,
                       'category': 'tree',
                       'accuracy': None,
                       'name': 'model_base_0',
                       'model': {'probLeft': 0.6200012047881724,
                                 'probRight': 0.37999879521182756,
                                 'prediction': None,
                                 'isCategorical': False,
                                 'feature': np.int64(0),
                                 'split': np.float64(3073.5),
                                 'pathProb': 1,
                                 'numSamples': 116203,
                                 'rightChild': {'probLeft': 0.7977670584505288,
                                                '